# Milestone 4 — VLM Membership Inference Attack

In [ ]:
import sys
sys.path.insert(0, '/content/milestone4')

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/milestone4/'

In [ ]:
!pip install -q xgboost

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')
import torch

from models import load_data, load_model_and_processor
from batches import process_dataset_in_batches
from classifier import build_membership_classifier, generate_submission

os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Always load splits — needed for labels and test IDs even when using cached features.
splits = load_data()

## 1. Extract Features

Run the cells in this section to extract features from scratch.
Skip to **Section 2** if you already have cached CSVs on Drive.

In [ ]:
processor, model = load_model_and_processor(device)

In [ ]:
train_features = process_dataset_in_batches(
    model, processor, splits['train'], has_labels=True, batch_size=200
)
train_features.to_csv(f'{SAVE_DIR}train.csv', index=False)
print('Train features saved.')

In [ ]:
val_features = process_dataset_in_batches(
    model, processor, splits['validation'], has_labels=True, batch_size=200
)
val_features.to_csv(f'{SAVE_DIR}val.csv', index=False)
print('Val features saved.')

In [ ]:
test_features = process_dataset_in_batches(
    model, processor, splits['test'], has_labels=False, batch_size=200
)
test_features.to_csv(f'{SAVE_DIR}test.csv', index=False)
print('Test features saved.')

## 2. Load Cached Features

Run this cell instead of Section 1 when resuming a session.

In [ ]:
# import pandas as pd
# train_features = pd.read_csv(f'{SAVE_DIR}train.csv')
# val_features   = pd.read_csv(f'{SAVE_DIR}val.csv')
# test_features  = pd.read_csv(f'{SAVE_DIR}test.csv')
# print('Features loaded.')

## 3. Train Classifier & Evaluate

In [ ]:
results = build_membership_classifier(train_features, val_features)

print('\n── Summary ──')
for name, r in results.items():
    print(f'{name:25s}  AUC={r["auc"]:.4f}  TPR@FPR=0.1={r["tpr_at_fpr01"]:.4f}')

best_name = max(results, key=lambda x: results[x]['auc'])
best      = results[best_name]
print(f'\nBest: {best_name}  AUC={best["auc"]:.4f}')

## 4. Generate Submission

In [ ]:
classifier_path = f'{SAVE_DIR}classifier.pkl'
with open(classifier_path, 'wb') as f:
    pickle.dump(best, f)
print(f'Classifier saved to {classifier_path}')

submission = generate_submission(
    splits['test'], test_features, best,
    output_path='submission.csv'
)
print(submission.head())

In [ ]:
from google.colab import files
files.download('submission.csv')